In [23]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib

In [24]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [25]:
query = """
SELECT DISTINCT c.CLICodigo, 
    c.CLINombres, 
    c.CLIApellidos,
    c.CLITelefonoCasa,
    c.CLICelular,
    c.CLIEmailPrincipal
    --c.CLITipoDocumento
FROM Clientes c
"""

# Ejecutar y cargar en DataFrame
df = pd.read_sql(query, engine)

In [26]:
#Limpiar Celular
def limpiar_celular(cel):
    if not cel:
        return ""
    cel = str(cel).strip().replace(" ", "").replace("-", "").replace("(", "").replace(")", "")
    if cel.startswith("+57"):
        cel = cel[3:]
    elif cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df['CLICelular'] = df['CLICelular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df['CelularValido'] = df['CLICelular'].apply(es_celular_valido)

#Limpiar y validar email
df['CLIEmailPrincipal'] = df['CLIEmailPrincipal'].apply(lambda x: str(x).strip().lower())

def es_email_valido(email):
    email = str(email).strip().lower()

    # Si está vacío
    if not email:
        return False
    
    # # Palabras prohibidas
    # palabras_no_permitidas = ["negad", "pendi"]
    # if any(palabra in email for palabra in palabras_no_permitidas):
    #     return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df['EmailValido'] = df['CLIEmailPrincipal'].apply(es_email_valido)

In [27]:
def limpiar_codigo(codigo):
    codigo = str(codigo).strip()     
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE)
    codigo = codigo.replace(' ', '')
    return codigo

df.insert(1, 'Documento', df['CLICodigo'].apply(limpiar_codigo))

df

,CLICodigo,Documento,CLINombres,CLIApellidos,CLITelefonoCasa,CLICelular,CLIEmailPrincipal,CelularValido,EmailValido
0,1000363796,1000363796,SARA DANIELA,PAZ MENDEZ,,3164766649,paz-sara@hotmail.com,True,True
1,1001391424,1001391424,VALENTINA,FRANCO,3002493141,3002493141,vale.franco1981@gmail.com,True,True
2,1002542697,1002542697,MARIA JULIANA,OROZCO GONZALEZ,,3127160903,julianaog13@gmail.com,True,True
3,1013260820,1013260820,MARIANA,DIAZ,3134495621,3134495621,maridiazd08@hotmail.com,True,True
4,1013644206,1013644206,None,,,,,False,False
...,...,...,...,...,...,...,...,...,...
381574,CYB6169725,YB6169725,CRISTOBAL ENRIQUE,ROMA GONZALEZ,1111111,1111111111,negado@provenzal.net,False,True
381575,Cyc004603,yc004603,SILVA,BRUNA,1111111,1111111111,pendiente@provenzal.net,False,True
381576,CYE065325,YE065325,LUIZ,GIANESELLA,UNDEFINED,UNDEFINED,lotaviogh@gmail.com,False,True
381577,CZ466154020000,Z466154020000,CARMEN ELIZABETH,ZACARIAS,5619327103,5619327103,zacariasbances@gmail.com,False,True


In [28]:
# # Detectar duplicados
# cel_duplicados = df['CLICelular'].duplicated(keep=False)
# email_duplicados = df['CLIEmailPrincipal'].duplicated(keep=False)

# # Marcar celulares duplicados como no válidos
# df.loc[cel_duplicados, 'CelularValido'] = False

# # Marcar correos duplicados como no válidos
# df.loc[email_duplicados, 'EmailValido'] = False

In [ ]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"
    
df['Contacto'] = df.apply(determinar_contacto, axis=1)

In [31]:
# # Exportar todos los registros con validaciones incluidas
df.to_excel("email_validos2.xlsx", index=False)